# Customer Feedback Analyzer

A CrewAI multi-agent workflow that analyzes customer reviews using custom tools:
- **Sentiment Analyzer Tool** — classifies feedback as positive / negative / neutral
- **Keyword Extractor Tool** — pulls recurring themes from feedback text
- **Action Item Generator Tool** — produces concrete next steps for the product team

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
from crewai.tools import BaseTool
from crewai import Agent, Task, Crew
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.3
)

In [ ]:
class SentimentAnalyzerTool(BaseTool):
    name: str = "Sentiment Analyzer Tool"
    description: str = "Classifies customer feedback as positive, negative, or neutral based on keyword cues."

    def _run(self, text: str) -> str:

        positive_words = ["love", "great", "excellent", "amazing", "smooth", "fast", "helpful", "easy"]
        negative_words = ["slow", "broken", "crash", "bug", "hate", "terrible", "confusing", "frustrating"]

        text_lower = text.lower()
        pos = sum(word in text_lower for word in positive_words)
        neg = sum(word in text_lower for word in negative_words)

        if pos > neg:
            sentiment = "Positive"
        elif neg > pos:
            sentiment = "Negative"
        else:
            sentiment = "Neutral"

        return f"Sentiment: {sentiment} (positive cues: {pos}, negative cues: {neg})"


class KeywordExtractorTool(BaseTool):
    name: str = "Keyword Extractor Tool"
    description: str = "Extracts recurring themes and keywords from customer feedback."

    def _run(self, text: str) -> str:

        themes = {
            "performance": ["slow", "fast", "lag", "speed", "loading"],
            "reliability": ["crash", "bug", "broken", "error", "freeze"],
            "usability": ["confusing", "easy", "intuitive", "hard", "simple"],
            "support": ["helpful", "unhelpful", "agent", "response", "chat"],
            "pricing": ["expensive", "cheap", "price", "cost", "worth"]
        }

        text_lower = text.lower()
        found = []

        for theme, words in themes.items():
            if any(word in text_lower for word in words):
                found.append(theme)

        if not found:
            return "No dominant themes detected."

        return f"Themes detected: {', '.join(found)}"


class ActionItemGeneratorTool(BaseTool):
    name: str = "Action Item Generator Tool"
    description: str = "Generates a concrete action item list for the product team based on feedback themes."

    def _run(self, themes: str) -> str:

        actions = {
            "performance": "Profile slow endpoints and optimize page-load metrics.",
            "reliability": "Triage crash reports and add regression tests for top failures.",
            "usability": "Schedule UX review of confusing flows; consider onboarding tooltips.",
            "support": "Audit recent support transcripts; expand help center coverage.",
            "pricing": "Review pricing-page messaging and value-proposition copy."
        }

        themes_lower = themes.lower()
        items = [f"- {action}" for theme, action in actions.items() if theme in themes_lower]

        if not items:
            return "No specific action items — feedback is general."

        return "Action Items:\n" + "\n".join(items)

In [ ]:
customer_feedback = """
I've been using this product for two months now. The dashboard is great and the
onboarding was easy, but the app crashes whenever I try to export reports. Loading
the analytics page is also painfully slow. Support was helpful when I reached out,
but I shouldn't be needing them this often.
"""

analyst_agent = Agent(
    role="Customer Feedback Analyst",
    goal="Analyze customer feedback to determine sentiment and extract recurring themes",
    backstory="Experienced product analyst skilled at turning raw user feedback into structured insights",
    verbose=True,
    tools=[
        SentimentAnalyzerTool(),
        KeywordExtractorTool()
    ],
    llm=llm
)

strategist_agent = Agent(
    role="Product Strategy Specialist",
    goal="Translate feedback themes into concrete action items for the product team",
    backstory="Product manager focused on prioritizing improvements that move customer-satisfaction metrics",
    verbose=True,
    tools=[
        ActionItemGeneratorTool()
    ],
    llm=llm
)

analyze_task = Task(
    description=f"""
Analyze the following customer feedback. Use the Sentiment Analyzer Tool to classify
overall sentiment, then use the Keyword Extractor Tool to identify recurring themes.

FEEDBACK:
{customer_feedback}
""",
    agent=analyst_agent,
    expected_output="Sentiment classification and a list of detected themes"
)

action_task = Task(
    description="""
Using the themes identified by the analyst, generate a list of concrete action items
for the product team. Pass the themes string into the Action Item Generator Tool.
""",
    agent=strategist_agent,
    expected_output="Bulleted action item list for the product team",
    context=[analyze_task]
)

In [ ]:
crew = Crew(
    agents=[
        analyst_agent,
        strategist_agent
    ],
    tasks=[
        analyze_task,
        action_task
    ],
    verbose=True
)

result = crew.kickoff()

print(result)